In [1]:
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    StorageContext, 
    ServiceContext, 
    load_index_from_storage
)
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.llms.groq import Groq

/home/sarveshharikant/anaconda3/envs/LLM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# !pip install google-generativeai>=0.3.0 llama-index-embeddings-gemini


## DATA INGESTION

In [3]:
reader = SimpleDirectoryReader(input_dir="data/")
documents = reader.load_data(num_workers=4)

In [4]:
# !pip install torch torchvision torchaudio


In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [ ]:
from llama_index.embeddings.gemini import GeminiEmbedding
embed_model = GeminiEmbedding(model_name="models/embedding-001")

In [ ]:
splitter = SemanticSplitterNodeParser(
              buffer_size=1, 
              breakpoint_percentile_threshold=95, 
              embed_model=embed_model
           )

nodes = splitter.get_nodes_from_documents(documents, show_progress=True)

In [ ]:
from llama_index.core import Document
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
# !set 
# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=25, chunk_overlap=0),
        TitleExtractor(),
        # GeminiEmbedding(),
        
        
    ]
)

# run the pipeline
nodes = pipeline.run(documents=[Document.example()])

## pipeline 2

In [20]:
from llama_index.llms.groq import Groq
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

Settings.llm = Groq(model="llama3-70b-8192")
Settings.embed_model = GeminiEmbedding(model_name="models/embedding-001")
Settings.text_splitter = SentenceSplitter(chunk_size=1024)
Settings.chunk_size = 512
Settings.chunk_overlap = 20
Settings.transformations = [SentenceSplitter(chunk_size=1024)]
# maximum input size to the LLM
Settings.context_window = 4096

# number of tokens reserved for text generation.
Settings.num_output = 256

In [21]:
# !pip install llama_index.settings

In [30]:

from llama_index.core import StorageContext, ServiceContext, load_index_from_storage
from llama_index.core.callbacks.base import CallbackManager
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.llms.groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()
import chainlit as cl

 

llm = Groq(model="llama3-70b-8192", api_key="")

from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 3900
Settings.chunk_size = 512
Settings.chunk_overlap = 64


# a vector store index only needs an embed model
index = VectorStoreIndex.from_documents(
    documents, embed_model=embed_model,
)


query_engine = index.as_query_engine(llm=llm)



Batches: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


In [35]:
query = "what is the author of this book"
resp = query_engine.query(query)


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.59it/s]


2024-10-26 23:58:45 - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [36]:

print(resp.response)

The author of this book is Peter Thiel with Blake Masters.


In [14]:
# !pip install chainlit